In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

# ==========================================
# 06 - GENERIC GOLD LOAD
# ==========================================

configs = (
    spark.table("mortgage_catalog.metadata.gold_config")
    .filter("is_active = true")
    .collect()
)

for config in configs:

    source_table = config["source_table"]
    target_table = config["target_table"]
    table_type   = config["table_type"]
    primary_key  = config["primary_key"]

    source = f"mortgage_catalog.silver.{source_table}"
    target = f"mortgage_catalog.gold.{target_table}"

    print(f"\nProcessing {table_type}")
    print(f"{source} → {target}")

    # ----------------------------------
    # Check source exists
    # ----------------------------------

    if not spark.catalog.tableExists(source):
        print(f"SKIPPED → {source} not found")
        continue

    source_df = spark.table(source)

    # ----------------------------------
    # For SCD2 take current record only
    # ----------------------------------

    if "is_current" in source_df.columns:

        source_df = source_df.filter(
            col("is_current") == True
        )

    # ----------------------------------
    # Initial Gold Load
    # ----------------------------------

    if not spark.catalog.tableExists(target):

        (
            source_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target)
        )

        print(f"CREATED → {target}")

    # ----------------------------------
    # Incremental Gold MERGE
    # ----------------------------------

    else:

        gold_delta = DeltaTable.forName(
            spark,
            target
        )

        (
            gold_delta.alias("T")
            .merge(
                source_df.alias("S"),
                f"T.{primary_key} = S.{primary_key}"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

        print(f"MERGED → {target}")


print("\nALL GOLD TABLES LOADED ✅")


Processing DIMENSION
mortgage_catalog.silver.property_current → mortgage_catalog.gold.dim_property
MERGED → mortgage_catalog.gold.dim_property

Processing DIMENSION
mortgage_catalog.silver.branch_history → mortgage_catalog.gold.dim_branch
MERGED → mortgage_catalog.gold.dim_branch

Processing DIMENSION
mortgage_catalog.silver.credit_score_current → mortgage_catalog.gold.dim_credit_score
MERGED → mortgage_catalog.gold.dim_credit_score

Processing FACT
mortgage_catalog.silver.loan_application_current → mortgage_catalog.gold.fact_loan
MERGED → mortgage_catalog.gold.fact_loan

Processing FACT
mortgage_catalog.silver.payment_current → mortgage_catalog.gold.fact_payment
MERGED → mortgage_catalog.gold.fact_payment

Processing DIMENSION
mortgage_catalog.silver.customer_current → mortgage_catalog.gold.dim_customer
MERGED → mortgage_catalog.gold.dim_customer

ALL GOLD TABLES LOADED ✅
